In [4]:
import os
from bisect import bisect_right
from collections import defaultdict

import numpy as np
import pandas as pd

# ----------------- USER SETTINGS -----------------
FOLDER = "/root/geant4/detector/Lung_ICRP/AK02_Lung"
ROOT_FILE = "Lung_AK02_400.root"   # One file containing ALL 36 detectors
TREE_NAME = "t"
EXCEL_FILE = "Lung_AK02_36_detectors_no_angle_cut.xlsx"

E0 = 662.0     # keV
ME = 511.0     # keV
BEAM = np.array([0.0, -1.0, 0.0])  # /gps/direction 0 -1 0
SCATTER_VOLUMES = (3, 5)
DETECTORS = list(range(400, 751, 10))  # 400,410,...,750 (36)
GAMMA = 22
COMPTON = 2013
# IMPORTANT: No theta_min, theta_max, or angle-agreement cut.
# -------------------------------------------------

FIELDS = ["pdg", "vlm", "trk", "pro", "stp", "k", "et", "de",
          "x", "y", "z", "px", "py", "pz"]
COLUMNS = [
    "tree_entry", "trk", "detector_volume", "scatter_volume",
    "scatter_vector_index", "scatter_pro", "scatter_stp",
    "scatter_k_keV", "scatter_energy_loss_keV", "scatter_et", "scatter_de",
    "scatter_x", "scatter_y", "scatter_z", "scatter_px", "scatter_py", "scatter_pz",
    "theta_p_scatter_deg", "theta_k_scatter_deg",
    "detector_vector_index", "detector_pro", "detector_stp",
    "detector_k_keV", "detector_et", "detector_de",
    "detector_x", "detector_y", "detector_z",
    "detector_px", "detector_py", "detector_pz",
    "theta_p_detector_deg", "theta_k_detector_deg",
    "theta_difference_scatter_deg", "theta_difference_detector_deg"
]


def angle_from_p(r):
    """Angle of recorded momentum with incident (0,-1,0), degrees."""
    p = np.array([r["px"], r["py"], r["pz"]], dtype=float)
    length = np.linalg.norm(p)
    if not np.isfinite(length) or length == 0:
        return np.nan
    return float(np.degrees(np.arccos(np.clip(np.dot(p, BEAM) / length, -1, 1))))


def angle_from_k(energy):
    """Single-scatter-equivalent angle (diagnostic, NOT a selection)."""
    if not np.isfinite(energy) or energy <= 0:
        return np.nan
    cos_theta = 1 - ME * (1 / energy - 1 / E0)
    if not -1 <= cos_theta <= 1:
        return np.nan
    return float(np.degrees(np.arccos(cos_theta)))


def delta(a, b):
    return a - b if np.isfinite(a) and np.isfinite(b) else np.nan


def read_record(event, index):
    result = {"index": index}
    for key in FIELDS:
        vec = getattr(event, key, None)
        value = vec[index] if vec is not None and index < len(vec) else None
        if key in ("pdg", "vlm", "trk", "pro", "stp"):
            result[key] = int(value) if value is not None else -1
        else:
            result[key] = float(value) if value is not None else np.nan
    return result


def matched_row(entry, s, d):
    ts, tk = angle_from_p(s), angle_from_k(s["k"])
    td, tdk = angle_from_p(d), angle_from_k(d["k"])
    return {
        "tree_entry": entry, "trk": s["trk"],
        "detector_volume": d["vlm"], "scatter_volume": s["vlm"],
        "scatter_vector_index": s["index"],
        "scatter_pro": s["pro"], "scatter_stp": s["stp"],
        "scatter_k_keV": s["k"],
        "scatter_energy_loss_keV": E0 - s["k"] if np.isfinite(s["k"]) else np.nan,
        "scatter_et": s["et"], "scatter_de": s["de"],
        **{"scatter_" + k: s[k] for k in ("x", "y", "z", "px", "py", "pz")},
        "theta_p_scatter_deg": ts, "theta_k_scatter_deg": tk,
        "detector_vector_index": d["index"],
        "detector_pro": d["pro"], "detector_stp": d["stp"],
        "detector_k_keV": d["k"], "detector_et": d["et"],
        "detector_de": d["de"],
        **{"detector_" + k: d[k] for k in ("x", "y", "z", "px", "py", "pz")},
        "theta_p_detector_deg": td, "theta_k_detector_deg": tdk,
        "theta_difference_scatter_deg": delta(ts, tk),
        "theta_difference_detector_deg": delta(td, tdk),
    }


def analyze(tree):
    detailed = []
    counts = {(det, vol): set() for det in DETECTORS for vol in SCATTER_VOLUMES}
    seen_in_detector = {det: set() for det in DETECTORS}
    records_in_detector = {det: 0 for det in DETECTORS}
    scatters_in_volume = {vol: set() for vol in SCATTER_VOLUMES}

    for entry, event in enumerate(tree):
        scatter_by_track = defaultdict(list)
        detector_by_track = defaultdict(lambda: defaultdict(list))
        n = min(len(event.pdg), len(event.vlm), len(event.trk))

        for i in range(n):
            if int(event.pdg[i]) != GAMMA:
                continue
            vol = int(event.vlm[i])
            if vol not in SCATTER_VOLUMES and vol not in DETECTORS:
                continue
            r = read_record(event, i)
            trk = r["trk"]
            if trk < 0:
                continue
            uid = (entry, trk)  # Unique gamma track WITHIN its event

            if vol in SCATTER_VOLUMES and r["pro"] == COMPTON:
                scatter_by_track[trk].append(r)
                scatters_in_volume[vol].add(uid)
            if vol in DETECTORS:
                detector_by_track[trk][vol].append(r)
                seen_in_detector[vol].add(uid)
                records_in_detector[vol] += 1

        # Match Compton records to LATER detector records of SAME track.
        for trk, scatters in scatter_by_track.items():
            for det_id, hits in detector_by_track.get(trk, {}).items():
                hit_indices = [hit["index"] for hit in hits]
                for s in scatters:
                    j = bisect_right(hit_indices, s["index"])
                    if j >= len(hits):
                        continue
                    d = hits[j]  # First later recorded step in detector
                    detailed.append(matched_row(entry, s, d))
                    counts[(det_id, s["vlm"])].add((entry, trk))

    return detailed, counts, seen_in_detector, records_in_detector, scatters_in_volume


def export_sheet(writer, df, name):
    """Split sheets exceeding Excel's 1,048,576-row limit."""
    limit = 1_048_575  # Leave one row for column headers
    if df.empty:
        df.to_excel(writer, sheet_name=name, index=False, freeze_panes=(1, 0))
    else:
        for part, start in enumerate(range(0, len(df), limit), 1):
            tab = name if part == 1 else f"{name}_{part}"
            df.iloc[start:start + limit].to_excel(
                writer, sheet_name=tab, index=False, freeze_panes=(1, 0))


def main():
    import ROOT  # PyROOT is required in your Geant4/ROOT environment

    full_path = os.path.join(FOLDER, ROOT_FILE)
    if not os.path.isfile(full_path):
        raise FileNotFoundError(full_path)
    root_file = ROOT.TFile.Open(full_path)
    if not root_file or root_file.IsZombie():
        raise OSError("Cannot open ROOT file: " + full_path)
    try:
        tree = root_file.Get(TREE_NAME)
        if not tree:
            raise KeyError("TTree '%s' not found" % TREE_NAME)
        print("ROOT file:", full_path)
        print("Events:", tree.GetEntries())
        print("Detector IDs: 400, 410, ..., 750")
        print("Angular restriction: NONE")
        detailed, counts, seen, records, scattered = analyze(tree)
    finally:
        root_file.Close()

    df = pd.DataFrame(detailed, columns=COLUMNS).sort_values(
        ["detector_volume", "tree_entry", "trk", "scatter_vector_index"],
        kind="stable"
    ).reset_index(drop=True)

    summary_rows = []
    for det in DETECTORS:
        s3, s5 = counts[(det, 3)], counts[(det, 5)]
        dr = df[df["detector_volume"] == det]
        summary_rows.append({
            "detector_volume": det,
            "gamma_from_vlm3": len(s3),
            "gamma_from_vlm5": len(s5),
            "gamma_from_both_3_and_5": len(s3 & s5),
            "unique_gamma_from_3_or_5": len(s3 | s5),
            "matched_scatter_records": len(dr),
            "gamma_tracks_any_origin_in_detector": len(seen[det]),
            "gamma_step_records_in_detector": records[det]
        })
    summary = pd.DataFrame(summary_rows)
    summary.loc[len(summary)] = {
        "detector_volume": "TOTAL detector-track pairs",
        **{c: int(summary[c].sum()) for c in summary.columns if c != "detector_volume"}
    }

    across3 = set().union(*(counts[(d, 3)] for d in DETECTORS))
    across5 = set().union(*(counts[(d, 5)] for d in DETECTORS))
    info = pd.DataFrame([
        ("ROOT input", ROOT_FILE),
        ("Number of detectors", len(DETECTORS)),
        ("Angular selection", "NONE"),
        ("All matched scatter records", len(df)),
        ("Unique detected gamma tracks from vlm 3", len(across3)),
        ("Unique detected gamma tracks from vlm 5", len(across5)),
        ("Unique detected gamma tracks from 3 OR 5", len(across3 | across5)),
        ("Scattered gamma tracks in vlm 3, detected or not", len(scattered[3])),
        ("Scattered gamma tracks in vlm 5, detected or not", len(scattered[5])),
        ("Counting", "Unique (tree_entry, trk) per detector and scatter volume"),
        ("Overlap rule", "A gamma scattered in BOTH 3 and 5 appears in both columns; union counted once"),
        ("Record order", "Assumes ROOT step vector indices are chronological within each event")
    ], columns=["Metric", "Value"])

    output = os.path.join(FOLDER, EXCEL_FILE)
    with pd.ExcelWriter(output, engine="openpyxl") as writer:
        export_sheet(writer, summary, "Summary_Counts")
        export_sheet(writer, info, "Run_Overview")
        export_sheet(writer, df, "All_matches")
        export_sheet(writer, df[df.scatter_volume == 3], "Scatter_Volume_3")
        export_sheet(writer, df[df.scatter_volume == 5], "Scatter_Volume_5")
        for det in DETECTORS:
            export_sheet(writer, df[df.detector_volume == det], f"Detector_{det}")

    print("\nSummary: gamma tracks that scattered in vlm 3 or 5 then reached detector")
    print(summary.to_string(index=False))
    print("\nGlobal unique detected gamma tracks (vlm 3 OR 5):", len(across3 | across5))
    print("Saved:", output)


if __name__ == "__main__":
    main()


ROOT file: /root/geant4/detector/Lung_ICRP/AK02_Lung/Lung_AK02_400.root
Events: 1000000
Detector IDs: 400, 410, ..., 750
Angular restriction: NONE

Summary: gamma tracks that scattered in vlm 3 or 5 then reached detector
           detector_volume  gamma_from_vlm3  gamma_from_vlm5  gamma_from_both_3_and_5  unique_gamma_from_3_or_5  matched_scatter_records  gamma_tracks_any_origin_in_detector  gamma_step_records_in_detector
                       400              130              654                        6                       778                      812                               907364                         1229127
                       410                0                0                        0                         0                        0                                    0                               0
                       420                0                0                        0                         0                        0                         